# Autonomous Commander — Multi-Agent Incident Investigation

This notebook walks through the full workflow:
1. Commander Agent evaluates an alert and builds an investigation plan
2. Three specialized agents run in parallel (Logs, Metrics, Deploy)
3. Commander synthesizes findings into a Root Cause Analysis report

```
ALERT → Commander → [Logs Agent | Metrics Agent | Deploy Agent] → Synthesis → RCA Report
```

## 1. Setup & Imports

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath('.'))

from graph.workflow import build_graph, run_investigation
from graph.visualize import print_ascii_graph

print('Imports OK')

## 2. Workflow Architecture Diagram

In [ ]:
print_ascii_graph()

## 3. LangGraph — Compiled Graph Visualization

In [ ]:
from IPython.display import Image, display

app = build_graph()

try:
    # Renders Mermaid diagram inline
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f'Mermaid render requires internet/graphviz: {e}')
    print('Mermaid source:')
    print(app.get_graph().draw_mermaid())

## 4. Define a Sample Alert

In [ ]:
alert = {
    'alert_id': 'ALT-20260325-001',
    'service': 'payment-service',
    'environment': 'production',
    'error_type': 'HighErrorRate',
    'error_message': 'NullPointerException in PaymentProcessor — circuit breaker OPEN',
    'severity_hint': 'P1',
    'timestamp': '2026-03-25T10:23:00Z',
    'time_range': 'last 30 minutes',
    'affected_endpoints': ['/api/v1/payments', '/api/v1/checkout'],
    'error_rate_percent': 22.1,
    'source': 'CloudWatch Alarm',
}

print(json.dumps(alert, indent=2))

## 5. Run the Autonomous Investigation

In [ ]:
# This invokes: Commander → Logs + Metrics + Deploy (parallel) → Synthesis
result = run_investigation(alert)
print('Investigation complete.')

## 6. Investigation Plan (Commander Output)

In [ ]:
print('Investigation Plan:')
for i, step in enumerate(result['investigation_plan'], 1):
    print(f'  {i}. {step}')

## 7. Logs Agent Findings

In [ ]:
print(json.dumps(result['logs_findings'], indent=2))

## 8. Metrics Agent Findings

In [ ]:
print(json.dumps(result['metrics_findings'], indent=2))

## 9. Deploy Intelligence Findings

In [ ]:
print(json.dumps(result['deploy_findings'], indent=2))

## 10. Final RCA Report

In [ ]:
print(result['final_report'])

## 11. Metrics Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Simulated telemetry for visualization
time_points = [f'T-{35 - i*5}m' for i in range(8)]
cpu = [45, 48, 52, 89, 95, 98, 97, 94]
latency = [120, 125, 130, 890, 1200, 1450, 1380, 1100]
error_rate = [0.1, 0.1, 0.2, 5.4, 18.2, 22.1, 19.8, 15.3]

fig, axes = plt.subplots(3, 1, figsize=(12, 10))
fig.suptitle('payment-service — Incident Telemetry', fontsize=14, fontweight='bold')

axes[0].plot(time_points, cpu, 'r-o', linewidth=2)
axes[0].axhline(y=80, color='orange', linestyle='--', label='Threshold 80%')
axes[0].set_ylabel('CPU %')
axes[0].set_title('CPU Utilization')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(time_points, latency, 'b-o', linewidth=2)
axes[1].axhline(y=500, color='orange', linestyle='--', label='SLA 500ms')
axes[1].set_ylabel('Latency p99 (ms)')
axes[1].set_title('Latency p99')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(time_points, error_rate, 'g-o', linewidth=2)
axes[2].axhline(y=1.0, color='orange', linestyle='--', label='Threshold 1%')
axes[2].set_ylabel('Error Rate %')
axes[2].set_title('Error Rate')
axes[2].set_xlabel('Time')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

# Mark deployment event
for ax in axes:
    ax.axvline(x=1, color='purple', linestyle=':', alpha=0.7, label='Deploy v2.4.1')

plt.tight_layout()
plt.savefig('incident_telemetry.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to incident_telemetry.png')